In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from sklearn.covariance import MinCovDet
import os
import time
from sklearn.utils.extmath import fast_logdet

In [2]:
import numpy as np
import pandas as pd
import hdbscan
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd 
import os

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA  # <-- Missing import
import matplotlib.pyplot as plt
#data load
raw_counts = pd.read_csv("gene_counts _for_ml.csv", index_col=0)

In [3]:
raw_counts.columns = raw_counts.columns.str.replace(r'^(CONTROL|TREATMENT).*', r'\1', regex=True)
print("\nCleaned columns:")
print(raw_counts.columns)


Cleaned columns:
Index(['TREATMENT', 'TREATMENT', 'TREATMENT', 'TREATMENT', 'CONTROL',
       'CONTROL', 'CONTROL', 'CONTROL'],
      dtype='object')


In [4]:
filtered_df = raw_counts

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf
import hdbscan
from scipy import linalg
from scipy.spatial.distance import pdist, squareform

def fast_logdet(matrix):
    """Compute log determinant efficiently"""
    try:
        return 2 * np.sum(np.log(np.diag(linalg.cholesky(matrix))))
    except:
        sign, logdet = np.linalg.slogdet(matrix)
        return logdet

# Create output directory
output_dir = "final_results/complete_data_distance_comparison"
os.makedirs(output_dir, exist_ok=True)

# 1. Data preparation
normalized_data = np.log2(filtered_df + 1)
normalized_data = pd.DataFrame(
    StandardScaler().fit_transform(normalized_data),
    index=normalized_data.index,
    columns=normalized_data.columns
)

print(f"Data shape: {normalized_data.shape}")
print(f"Genes: {normalized_data.shape[0]}, Samples: {normalized_data.shape[1]}")

# ----------------------------
# NUMERICALLY STABLE Mahalanobis for few samples
# ----------------------------
class StableMahalanobisHDBSCAN:
    def __init__(self, min_cluster_size=5, min_samples=None):
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        
    def _stable_mahalanobis(self, X):
        """Numerically stable Mahalanobis distance calculation for few samples"""
        try:
            n_genes, n_samples = X.shape
            print(f"Calculating Mahalanobis for {n_genes} genes with {n_samples} samples")
            
            # 1. Use Ledoit-Wolf shrinkage for stable covariance estimation
            cov_estimator = LedoitWolf().fit(X)
            cov = cov_estimator.covariance_
            
            # 2. Enhanced regularization for numerical stability
            trace_cov = np.trace(cov)
            if trace_cov == 0:
                trace_cov = 1.0  # Avoid division by zero
                
            # Stronger regularization for few samples
            reg_strength = 1e-4 * trace_cov / n_samples
            reg = reg_strength * np.eye(n_samples)
            cov_reg = cov + reg
            
            # 3. Check condition number
            cond_number = np.linalg.cond(cov_reg)
            print(f"Covariance matrix condition number: {cond_number:.2e}")
            
            # 4. Use pseudo-inverse instead of regular inverse for stability
            if cond_number > 1e12:
                print("Using pseudo-inverse for numerical stability")
                cov_inv = np.linalg.pinv(cov_reg)
            else:
                cov_inv = np.linalg.inv(cov_reg)
            
            # 5. Memory-efficient distance calculation
            print("Computing pairwise distances...")
            dist_vector = pdist(X, 'mahalanobis', VI=cov_inv)
            dist_matrix = squareform(dist_vector)
            np.fill_diagonal(dist_matrix, 0)
            
            return dist_matrix
            
        except Exception as e:
            print(f"Stable Mahalanobis calculation failed: {str(e)}")
            import traceback
            traceback.print_exc()
            return None

    def fit(self, X):
        dist_matrix = self._stable_mahalanobis(X)
        if dist_matrix is None:
            print("Using Euclidean as fallback for Mahalanobis")
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='euclidean'
            )
            self.labels_ = self.clusterer.fit_predict(X)
        else:
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='precomputed'
            )
            self.labels_ = self.clusterer.fit_predict(dist_matrix)
        return self

# ----------------------------
# Alternative: PCA-based Mahalanobis (even more stable)
# ----------------------------
class PCAMahalanobisHDBSCAN:
    def __init__(self, min_cluster_size=5, min_samples=None, n_components=None):
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        self.n_components = n_components
        
    def _pca_mahalanobis(self, X):
        """Mahalanobis on PCA-reduced data for maximum stability"""
        try:
            n_genes, n_samples = X.shape
            
            # Determine optimal number of PCA components
            if self.n_components is None:
                n_components = min(n_samples - 1, 6)  # Leave at least 1 DOF
            else:
                n_components = min(self.n_components, n_samples - 1)
            
            print(f"Using PCA with {n_components} components for stability")
            pca = PCA(n_components=n_components)
            X_reduced = pca.fit_transform(X)
            
            # Now compute Mahalanobis on reduced data
            cov_estimator = LedoitWolf().fit(X_reduced)
            cov = cov_estimator.covariance_
            
            # Regularization
            trace_cov = np.trace(cov)
            reg_strength = 1e-6 * trace_cov / n_components
            reg = reg_strength * np.eye(n_components)
            cov_reg = cov + reg
            
            cov_inv = np.linalg.inv(cov_reg)
            
            # Compute distances
            dist_vector = pdist(X_reduced, 'mahalanobis', VI=cov_inv)
            dist_matrix = squareform(dist_vector)
            np.fill_diagonal(dist_matrix, 0)
            
            return dist_matrix
            
        except Exception as e:
            print(f"PCA Mahalanobis failed: {str(e)}")
            return None

    def fit(self, X):
        dist_matrix = self._pca_mahalanobis(X)
        if dist_matrix is None:
            print("Using Euclidean as fallback for PCA Mahalanobis")
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='euclidean'
            )
            self.labels_ = self.clusterer.fit_predict(X)
        else:
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='precomputed'
            )
            self.labels_ = self.clusterer.fit_predict(dist_matrix)
        return self

# ----------------------------
# Metric Comparison
# ----------------------------
metrics = [
    ('euclidean', hdbscan.HDBSCAN(metric='euclidean')),
    ('manhattan', hdbscan.HDBSCAN(metric='manhattan')),
    ('correlation', hdbscan.HDBSCAN(metric='correlation')),
    ('mahalanobis_stable', StableMahalanobisHDBSCAN()),
    ('mahalanobis_pca', PCAMahalanobisHDBSCAN(n_components=6))
]

results = []
for name, clusterer in metrics:
    print(f"\n{'='*40}\nClustering with: {name.upper()}\n{'='*40}")
    
    try:
        start_time = time.time()
        
        if 'mahalanobis' in name:
            print(f"Using {name} approach...")
            clusterer.fit(normalized_data.values)
            clusters = clusterer.labels_
        else:
            clusters = clusterer.fit_predict(normalized_data.values)
            
        # Analysis
        unique_clusters = np.unique(clusters)
        n_clusters = len(unique_clusters) - 1
        noise_ratio = np.mean(clusters == -1)
        cluster_sizes = [np.sum(clusters == c) for c in unique_clusters if c != -1]
        
        results.append({
            'metric': name,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'clusters': clusters,
            'cluster_sizes': cluster_sizes,
            'time': time.time() - start_time
        })
        
        print(f"Number of clusters: {n_clusters}")
        print(f"Noise ratio: {noise_ratio:.1%}")
        print(f"Cluster sizes: {sorted(cluster_sizes, reverse=True)[:10]}")
        print(f"Time elapsed: {results[-1]['time']:.2f} seconds")
        
    except Exception as e:
        print(f"Clustering failed: {str(e)}")
        import traceback
        traceback.print_exc()
        results.append({
            'metric': name,
            'error': str(e),
            'clusters': None
        })

# ----------------------------
# Visualization and Saving Results
# ----------------------------
# (Keep the visualization and saving code from previous versions)
print("Computing PCA for visualization...")
pca = PCA(n_components=2).fit_transform(normalized_data.values)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, result in enumerate(results):
    ax = axes[i]
    
    if result['clusters'] is None:
        ax.text(0.5, 0.5, f"Failed:\n{result.get('error','')}", 
                ha='center', va='center', fontsize=10)
        ax.set_title(f"{result['metric'].upper()} - FAILED", fontsize=12)
        continue
        
    sc = ax.scatter(
        pca[:, 0], pca[:, 1],
        c=result['clusters'],
        cmap='tab20',
        alpha=0.7,
        s=10
    )
    
    title = (
        f"{result['metric'].upper()}\n"
        f"Clusters: {result['n_clusters']} | "
        f"Noise: {result['noise_ratio']:.1%}\n"
        f"Time: {result['time']:.2f}s"
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    
    plt.colorbar(sc, ax=ax, label='Cluster ID')

# Hide unused subplots
for i in range(len(results), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig(f"{output_dir}/clustering_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

# Save results
for result in results:
    if result['clusters'] is not None:
        pd.DataFrame({
            'gene': normalized_data.index,
            'cluster': result['clusters']
        }).to_csv(f"{output_dir}/clusters_{result['metric']}.csv", index=False)

summary_data = []
for r in results:
    summary_data.append({
        'metric': r['metric'],
        'n_clusters': r.get('n_clusters', np.nan),
        'noise_ratio': r.get('noise_ratio', np.nan),
        'largest_cluster': max(r['cluster_sizes']) if 'cluster_sizes' in r and r['cluster_sizes'] else np.nan,
        'time_seconds': r.get('time', np.nan),
        'status': 'SUCCESS' if r['clusters'] is not None else 'FAILED'
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(f"{output_dir}/summary_stats.csv", index=False)

print(f"\nAnalysis complete. Results saved to: {output_dir}")
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))

Data shape: (28581, 8)
Genes: 28581, Samples: 8

Clustering with: EUCLIDEAN


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 102
Noise ratio: 30.0%
Cluster sizes: [np.int64(12924), np.int64(5400), np.int64(127), np.int64(123), np.int64(92), np.int64(87), np.int64(74), np.int64(71), np.int64(70), np.int64(29)]
Time elapsed: 1.99 seconds

Clustering with: MANHATTAN


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 88
Noise ratio: 17.8%
Cluster sizes: [np.int64(16594), np.int64(5400), np.int64(127), np.int64(123), np.int64(92), np.int64(87), np.int64(74), np.int64(71), np.int64(70), np.int64(29)]
Time elapsed: 1.69 seconds

Clustering with: CORRELATION


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 186
Noise ratio: 70.1%
Cluster sizes: [np.int64(5400), np.int64(620), np.int64(127), np.int64(123), np.int64(97), np.int64(92), np.int64(88), np.int64(74), np.int64(71), np.int64(70)]
Time elapsed: 15.02 seconds

Clustering with: MAHALANOBIS_STABLE
Using mahalanobis_stable approach...
Calculating Mahalanobis for 28581 genes with 8 samples
Covariance matrix condition number: 6.95e+02
Computing pairwise distances...
Number of clusters: 97
Noise ratio: 33.7%
Cluster sizes: [np.int64(11938), np.int64(5400), np.int64(127), np.int64(123), np.int64(92), np.int64(87), np.int64(74), np.int64(71), np.int64(70), np.int64(30)]
Time elapsed: 26.83 seconds

Clustering with: MAHALANOBIS_PCA
Using mahalanobis_pca approach...
Using PCA with 6 components for stability
Number of clusters: 105
Noise ratio: 38.4%
Cluster sizes: [np.int64(10581), np.int64(5401), np.int64(127), np.int64(127), np.int64(93), np.int64(90), np.int64(75), np.int64(71), np.int64(71), np.int64(29)]
Time elapsed:

Exception ignored in: <function ResourceTracker.__del__ at 0x73d87a583920>
Traceback (most recent call last):
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x757fc1d83920>
Traceback (most recent call last):
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in